# 08 · HDFS + YARN Bootstrap (Case C)

**Theory**: docs/07-spark-and-hdfs.md

**Prerequisite**: `make up-hadoop` (NameNode, 2 DataNodes, ResourceManager,
2 NodeManagers, and the HttpFS gateway — ~6GB RAM recommended).

This is the first notebook where the Driver (this process, on your host)
submits work to a **real distributed cluster manager (YARN)**, not Spark's
own Standalone manager. Read docs/07 first — in particular, why we use
`webhdfs://localhost:14000/...` here instead of the `hdfs://namenode:8020/...`
scheme the theory slides show.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, "../scripts")
from lab_utils import layer_path, upload_bronze_table_to_hdfs
from pyspark.sql import SparkSession

# Case C: client mode against a dockerized YARN cluster (make up-hadoop).
# The driver runs on your HOST (this process), but executors run inside the
# nodemanager1/nodemanager2 containers.

# Spark needs HADOOP_CONF_DIR to find the ResourceManager. This is a
# HOST-only config (localhost + published ports), separate from
# config/hadoop/ (used by the containers themselves) — see
# config/hadoop-client/ for why.
os.environ["HADOOP_CONF_DIR"] = str(Path("../config/hadoop-client").resolve())
# Without Kerberos, HDFS trusts whatever username the client claims. Your
# host OS username almost certainly is not "root" (the owner of "/" in this
# cluster, since every container in the "hadoop" profile runs as root) —
# this makes the driver identify as "root" too, avoiding a spurious
# AccessControlException on YARN's staging directory.
os.environ["HADOOP_USER_NAME"] = "root"

spark = (
    SparkSession.builder.appName("08-hdfs-yarn-bootstrap")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    # For executors to call back to this driver, docker-compose.yml gives
    # them host.docker.internal via Docker's host-gateway extra_hosts entry
    # — hence spark.driver.host below.
    .config("spark.driver.host", "host.docker.internal")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .config("spark.yarn.am.memory", "1g")
    # Use Spark's jars already bind-mounted into the NodeManager containers
    # (docker-compose.yml) instead of having YARN stage them through HDFS —
    # sidesteps the driver-vs-container hostname mismatch for the URI that
    # staging would otherwise bake in. See docs/07-spark-and-hdfs.md for why
    # this case also uses webhdfs:// instead of hdfs:// for I/O.
    .config("spark.yarn.jars", "local:/opt/spark-jars/*")
    .getOrCreate()
)
spark

## Your first YARN application

The moment you call an action below, watch http://localhost:8088 (the
ResourceManager UI) — you'll see a new Application appear, transition
through states (`ACCEPTED` -> `RUNNING`), and an Application Master get
scheduled onto one of the two NodeManagers.

In [ ]:
df = spark.range(0, 5_000_000)
print(f"Row count: {df.count():,}")
print("Check http://localhost:8088 for the Application that just ran.")

## Getting the Bronze layer into HDFS

Unlike Case B/D (where a shared Docker volume already makes `./data`
visible to the containers), Case C's NodeManagers have no access to your
host filesystem — so the first step is a one-time upload through the
HttpFS gateway. `upload_bronze_table_to_hdfs` walks the locally generated
Parquet files and writes them to HDFS via WebHDFS REST calls, preserving
the `ano=/mes=` partition folders.

In [ ]:
upload_bronze_table_to_hdfs("vendas")
upload_bronze_table_to_hdfs("empresas")
upload_bronze_table_to_hdfs("funcionarios")

In [ ]:
vendas_hdfs = spark.read.parquet(layer_path("hdfs", "bronze", "vendas"))
print(f"Read back from HDFS: {vendas_hdfs.count():,} rows")
vendas_hdfs.show(5)

## Visual check: the NameNode UI

Open http://localhost:9870 -> Utilities -> Browse the file system, and
navigate to `/datalake/bronze/vendas`. You'll see the `part-*.parquet` files
Spark wrote, replicated across the 2 DataNodes (replication factor 2, set in
`.env`).

In [ ]:
spark.stop()